# CrewAI Stock Picker: Structured Output, Custom Tool, Hierarchical Process, Memory

이번 노트북에서는 CrewAI의 고급 기능을 활용한 **Stock Picker** 프로젝트를 분석합니다. 이전 프로젝트(Debate, Financial Researcher)에서 다루지 않았던 핵심 개념들을 집중적으로 학습합니다.

## 개요

| 주제 | 내용 |
|------|------|
| Structured Output | Pydantic 모델로 태스크 출력을 구조화 |
| Custom Tool | PushNotificationTool — 실제 알림 전송 도구 |
| Hierarchical Process | Manager Agent가 태스크를 동적으로 위임 |
| Memory | Short-term, Long-term, Entity Memory 구성 및 용도 |

## 학습 목표

1. `output_pydantic`으로 태스크 출력을 구조화된 데이터로 받는 방법 이해하기
2. `BaseTool`을 상속하여 실제 외부 서비스와 연동하는 커스텀 도구 만들기
3. `Process.hierarchical`과 Manager Agent의 동작 방식 이해하기
4. CrewAI의 4가지 Memory 타입별 역할과 사용 이유 파악하기

---

## 이전 프로젝트와의 비교

| | Debate (03-2) | Financial Researcher (03-3) | **Stock Picker (이번)** |
|---|---|---|---|
| **에이전트 수** | 2 | 2 | **4** (finder, researcher, picker, manager) |
| **프로세스** | sequential | sequential | **hierarchical** |
| **도구** | 없음 | SerperDevTool | SerperDevTool + **PushNotificationTool** |
| **출력 형식** | 자유 텍스트 | 자유 텍스트 | **Pydantic 구조화 출력** |
| **메모리** | 없음 | 없음 | **Short/Long/Entity Memory** |

---

## 1. 프로젝트 구조

```
stock_picker/
├── pyproject.toml                  # crewai[tools]==1.6.1
├── .env                            # MODEL=gpt-4o-mini
├── knowledge/
│   └── user_preference.txt         # 사용자 선호 정보
├── output/                         # 태스크 결과물
│   ├── trending_companies.json     # 트렌딩 기업 목록
│   ├── research_report.json        # 기업 분석 리포트
│   └── decision.md                 # 최종 투자 결정
├── memory/                         # 메모리 저장소 (자동 생성)
│   └── long_term_memory_storage.db # SQLite 장기 메모리
└── src/stock_picker/
    ├── config/
    │   ├── agents.yaml
    │   └── tasks.yaml
    ├── tools/
    │   └── push_tool.py            # 푸시 알림 커스텀 도구
    ├── crew.py                     # 핵심: 구조화 출력 + 메모리 + hierarchical
    └── main.py
```

### 프로젝트의 아이디어

주어진 **섹터(sector)**에서:
1. **trending_company_finder**가 웹 검색으로 트렌딩 기업 2-3개 발굴
2. **financial_researcher**가 각 기업을 상세 분석
3. **stock_picker**가 최적의 투자 기업을 선정하고 푸시 알림 전송
4. **manager**가 전체 워크플로우를 동적으로 조율

---

## 2. Structured Output — 구조화된 출력

이전 프로젝트에서는 태스크 출력이 자유 텍스트였습니다. Stock Picker에서는 **Pydantic 모델**로 출력 형태를 엄격하게 정의합니다.

### 왜 구조화된 출력이 필요한가?

```
┌─────────────────────────────────────────────────────────────────────┐
│           자유 텍스트 출력 vs 구조화된 출력                         │
├──────────────────────────────┬──────────────────────────────────────┤
│     자유 텍스트 (이전)       │     Pydantic 모델 (이번)            │
├──────────────────────────────┼──────────────────────────────────────┤
│  "Apple is trending..."     │  {                                   │
│  형식이 매번 달라질 수 있음  │    "name": "Apple",                 │
│  파싱이 어렵고 불안정        │    "ticker": "AAPL",                │
│  다음 태스크가 해석에 의존   │    "reason": "AI chip investment"   │
│                              │  }                                   │
│                              │  타입 검증, 자동 파싱, 안정적       │
└──────────────────────────────┴──────────────────────────────────────┘
```

### Pydantic 모델 정의 (crew.py에서)

```python
from pydantic import BaseModel, Field
from typing import List

class TrendingCompany(BaseModel):
    """트렌딩 기업 1개"""
    name: str = Field(description="Company name")
    ticker: str = Field(description="Stock ticker symbol")
    reason: str = Field(description="Reason this company is trending")

class TrendingCompanyList(BaseModel):
    """트렌딩 기업 목록"""
    companies: List[TrendingCompany] = Field(
        description="List of companies trending in the news"
    )

class TrendingCompanyResearch(BaseModel):
    """기업 1개의 상세 분석"""
    name: str = Field(description="Company name")
    market_position: str = Field(description="Current market position")
    future_outlook: str = Field(description="Future outlook and growth")
    investment_potential: str = Field(description="Investment potential")

class TrendingCompanyResearchList(BaseModel):
    """전체 기업 분석 목록"""
    research_list: List[TrendingCompanyResearch] = Field(
        description="Comprehensive research on all trending companies"
    )
```

### Task에 연결하는 방법

```python
@task
def find_trending_companies(self) -> Task:
    return Task(
        config=self.tasks_config['find_trending_companies'],
        output_pydantic=TrendingCompanyList,    # ← 여기!
    )
```

| 속성 | 설명 |
|------|------|
| `output_pydantic` | 출력을 Pydantic 모델로 강제 — 자동 검증 + JSON 변환 |
| `output_file` | 결과를 파일로 저장 (`.json`이면 구조화된 JSON으로 저장) |

`output_pydantic`을 지정하면 CrewAI가 LLM 출력을 자동으로 파싱하여 Pydantic 객체로 변환합니다. 다음 태스크에서 `result.pydantic.companies[0].ticker` 같은 방식으로 접근할 수 있습니다.

---

## 3. Custom Tool — PushNotificationTool

Financial Researcher에서는 기존 제공 도구(SerperDevTool)를 사용했습니다. Stock Picker에서는 **외부 API와 연동하는 커스텀 도구**를 직접 만들어 사용합니다.

### push_tool.py 전체 코드

```python
from crewai.tools import BaseTool
from typing import Type
from pydantic import BaseModel, Field
import os
import requests


class PushNotification(BaseModel):
    """A message to be sent to the user"""
    message: str = Field(..., description="The message to be sent to the user.")

class PushNotificationTool(BaseTool):
    name: str = "Send a Push Notification"
    description: str = (
        "This tool is used to send a push notification to the user."
    )
    args_schema: Type[BaseModel] = PushNotification

    def _run(self, message: str) -> str:
        pushover_user = os.getenv("PUSHOVER_USER")
        pushover_token = os.getenv("PUSHOVER_TOKEN")
        pushover_url = "https://api.pushover.net/1/messages.json"

        print(f"Push: {message}")
        payload = {
            "user": pushover_user,
            "token": pushover_token,
            "message": message
        }
        requests.post(pushover_url, data=payload)
        return '{"notification": "ok"}'
```

### 구조 분석

```
┌─────────────────────────────────────────────────────────────────────┐
│              PushNotificationTool 동작 흐름                         │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│  stock_picker Agent                                                 │
│    │  "Apple을 투자 추천합니다" → 도구 호출 결정                    │
│    ▼                                                                │
│  PushNotificationTool._run(message="Apple 투자 추천")              │
│    │  1. 환경변수에서 PUSHOVER_USER, PUSHOVER_TOKEN 로드            │
│    │  2. Pushover API로 POST 요청                                   │
│    │  3. '{"notification": "ok"}' 반환                             │
│    ▼                                                                │
│  stock_picker Agent                                                 │
│    └─ 알림 전송 확인 후, 상세 리포트 작성 계속                      │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

### 이전 노트북의 WordCountTool과 비교

| | WordCountTool (03-1) | PushNotificationTool (이번) |
|---|---|---|
| **동작** | 텍스트 처리 (순수 함수) | 외부 API 호출 (사이드 이펙트) |
| **환경변수** | 불필요 | PUSHOVER_USER, PUSHOVER_TOKEN 필요 |
| **반환값** | 계산 결과 문자열 | API 응답 확인 JSON |
| **실제 사용** | 학습용 예제 | 프로덕션에서 실제 알림 전송 |

---

## 4. Hierarchical Process — Manager Agent가 조율하는 실행

이전 프로젝트들은 모두 `Process.sequential`(태스크를 순서대로 실행)이었습니다. Stock Picker는 **`Process.hierarchical`**을 사용하여 Manager Agent가 태스크를 동적으로 위임합니다.

### Sequential vs Hierarchical

```
┌─────────────────────────────────────────────────────────────────────┐
│  Sequential (이전 프로젝트들):                                      │
│                                                                     │
│    Task A ──▶ Task B ──▶ Task C                                     │
│    고정된 순서, 각 태스크가 미리 정해진 에이전트에게 할당            │
│                                                                     │
├─────────────────────────────────────────────────────────────────────┤
│  Hierarchical (Stock Picker):                                       │
│                                                                     │
│                   ┌──────────┐                                      │
│                   │ Manager  │  (gpt-4o)                            │
│                   │  Agent   │                                      │
│                   └────┬─────┘                                      │
│              ┌─────────┼─────────┐                                  │
│              ▼         ▼         ▼                                  │
│         ┌────────┐ ┌────────┐ ┌────────┐                           │
│         │ finder │ │research│ │ picker │                            │
│         └────────┘ └────────┘ └────────┘                           │
│                                                                     │
│    Manager가 상황에 따라 태스크를 위임하고 결과를 검토               │
│    필요하면 재할당하거나 추가 작업을 지시                            │
└─────────────────────────────────────────────────────────────────────┘
```

### crew.py에서의 설정

```python
@crew
def crew(self) -> Crew:
    manager = Agent(
        config=self.agents_config['manager'],
        allow_delegation=True               # 다른 에이전트에게 위임 가능
    )

    return Crew(
        agents=self.agents,
        tasks=self.tasks,
        process=Process.hierarchical,       # ← hierarchical!
        manager_agent=manager,              # ← Manager Agent 지정
        verbose=True,
    )
```

### Manager Agent (agents.yaml)

```yaml
manager:
  role: >
    Manager
  goal: >
    You are a skilled project manager who can delegate tasks
    in order to achieve your goal, which is to pick the best
    company for investment.
  backstory: >
    You are an experienced and highly effective project manager
    who can delegate tasks to the right people.
  llm: openai/gpt-4o     # ← 다른 에이전트보다 상위 모델 사용!
```

| 특성 | Sequential | Hierarchical |
|------|-----------|-------------|
| **태스크 할당** | 미리 고정 | Manager가 동적으로 위임 |
| **실행 순서** | tasks 리스트 순서 | Manager가 판단 |
| **에러 처리** | 실패 시 중단 | Manager가 재할당 가능 |
| **Manager Agent** | 불필요 | 필수 (`manager_agent=`) |
| **모델 비용** | 균일 | Manager에 상위 모델 사용 권장 |
| **적합한 상황** | 단순한 파이프라인 | 복잡한 의사결정, 동적 워크플로우 |

---

## 5. Memory — 에이전트의 기억 체계

이전 프로젝트에서는 메모리를 사용하지 않았습니다. Stock Picker는 **3가지 메모리 타입**을 모두 활성화하여 에이전트가 더 스마트하게 동작합니다.

### 왜 메모리가 필요한가?

메모리가 없으면 에이전트는 매 실행마다 **처음부터 시작**합니다:
- 같은 기업을 반복 추천
- 이전 분석 결과를 활용하지 못함
- 대화 중 앞서 언급한 정보를 잊음

### CrewAI의 4가지 Memory 타입

```
┌─────────────────────────────────────────────────────────────────────┐
│                    CrewAI Memory 체계                               │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│  ┌─────────────────┐  현재 실행 중의 대화/결과를 기억              │
│  │ Short-term       │  RAG 기반 벡터 저장소                        │
│  │ Memory           │  실행이 끝나면 사라짐                        │
│  └─────────────────┘  비유: 회의 중 화이트보드                     │
│                                                                     │
│  ┌─────────────────┐  실행 간(across sessions) 영구 저장           │
│  │ Long-term        │  SQLite DB에 저장                            │
│  │ Memory           │  이전 실행의 학습/결과를 기억                 │
│  └─────────────────┘  비유: 업무 노트/일지                         │
│                                                                     │
│  ┌─────────────────┐  핵심 엔티티(사람, 회사 등) 정보 추적         │
│  │ Entity           │  RAG 기반 벡터 저장소                        │
│  │ Memory           │  엔티티 간 관계와 속성을 기억                 │
│  └─────────────────┘  비유: 고객 DB / CRM                          │
│                                                                     │
│  ┌─────────────────┐  사용자의 선호도와 피드백을 학습              │
│  │ Contextual       │  (이 프로젝트에서는 미사용)                  │
│  │ Memory           │  반복 실행 시 사용자 맞춤 응답               │
│  └─────────────────┘  비유: 단골 고객 취향 파악                    │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

### 각 Memory 타입의 상세 비교

| Memory 타입 | 지속 기간 | 저장 방식 | 용도 | Stock Picker에서의 역할 |
|------------|----------|----------|------|------------------------|
| **Short-term** | 현재 실행 중만 | RAG (벡터) | 태스크 간 컨텍스트 공유 | finder 결과를 researcher가 참고 |
| **Long-term** | 영구 (세션 간) | SQLite DB | 과거 학습/경험 보존 | "이전에 Apple을 추천했으니 다른 기업 선정" |
| **Entity** | 현재 실행 중 | RAG (벡터) | 핵심 엔티티 정보 추적 | "Apple: 시가총액 3조, AI 칩 투자" 등 기업 정보 |
| **Contextual** | 영구 (세션 간) | - | 사용자 선호도 학습 | (이 프로젝트에서 미사용) |

### crew.py에서의 메모리 설정

```python
from crewai.memory import LongTermMemory, ShortTermMemory, EntityMemory
from crewai.memory.storage.rag_storage import RAGStorage
from crewai.memory.storage.ltm_sqlite_storage import LTMSQLiteStorage

@crew
def crew(self) -> Crew:
    return Crew(
        ...
        memory=True,                           # 메모리 전체 활성화

        # Long-term: SQLite에 영구 저장
        long_term_memory=LongTermMemory(
            storage=LTMSQLiteStorage(
                db_path="./memory/long_term_memory_storage.db"
            )
        ),

        # Short-term: RAG 벡터 저장소
        short_term_memory=ShortTermMemory(
            storage=RAGStorage(
                embedder_config={
                    "provider": "openai",
                    "config": {"model": "text-embedding-3-small"}
                },
                type="short_term",
                path="./memory/"
            )
        ),

        # Entity: RAG 벡터 저장소
        entity_memory=EntityMemory(
            storage=RAGStorage(
                embedder_config={
                    "provider": "openai",
                    "config": {"model": "text-embedding-3-small"}
                },
                type="short_term",
                path="./memory/"
            )
        ),
    )
```

### 메모리 동작 흐름 (Stock Picker 실행 시)

```
┌─────────────────────────────────────────────────────────────────────┐
│                 메모리가 있는 실행 흐름                              │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│  [1회차 실행]                                                       │
│  finder: "Apple, NVIDIA, Tesla가 트렌딩"                           │
│    → Short-term Memory에 저장 (현재 세션 내 공유)                   │
│    → Entity Memory에 각 기업 정보 저장                              │
│  researcher: Short-term에서 finder 결과 참조하여 분석               │
│  picker: "Apple 추천" → Long-term Memory에 저장                    │
│                                                                     │
│  [2회차 실행]                                                       │
│  finder: Long-term Memory 확인 → "Apple은 이미 추천했으니 제외"    │
│    → 새로운 기업 발굴                                               │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

에이전트 레벨에서도 `memory=True`를 설정하면 해당 에이전트가 메모리를 적극 활용합니다:

```python
@agent
def trending_company_finder(self) -> Agent:
    return Agent(
        config=self.agents_config['trending_company_finder'],
        tools=[SerperDevTool()],
        memory=True,          # ← 이 에이전트는 메모리를 적극 활용
    )
```

---

## 6. 전체 실행 흐름

### agents.yaml 요약

| Agent | role | llm | tools | memory |
|-------|------|-----|-------|--------|
| trending_company_finder | 뉴스에서 트렌딩 기업 발굴 | gpt-4o-mini | SerperDevTool | True |
| financial_researcher | 기업 상세 분석 | gpt-4o-mini | SerperDevTool | - |
| stock_picker | 최적 투자 기업 선정 + 알림 | gpt-4o-mini | PushNotificationTool | True |
| manager | 전체 워크플로우 조율 | **gpt-4o** | - | - |

### tasks.yaml 요약

| Task | output_pydantic | context | output_file |
|------|----------------|---------|-------------|
| find_trending_companies | TrendingCompanyList | - | trending_companies.json |
| research_trending_companies | TrendingCompanyResearchList | [find_trending_companies] | research_report.json |
| pick_best_company | - (자유 텍스트) | [research_trending_companies] | decision.md |

### main.py

```python
from datetime import datetime
from stock_picker.crew import StockPicker

def run():
    inputs = {
        'sector': 'Technology',
        'current_date': str(datetime.now())
    }
    result = StockPicker().crew().kickoff(inputs=inputs)
    print(result.raw)
```

### 전체 흐름 다이어그램

```
┌─────────────────────────────────────────────────────────────────────┐
│             Stock Picker 실행 흐름 (Hierarchical)                   │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│  inputs = {'sector': 'Technology', 'current_date': '...'}          │
│       │                                                             │
│       ▼                                                             │
│  ┌──────────────┐                                                   │
│  │   Manager     │  (gpt-4o) 전체 조율                              │
│  │   Agent       │                                                   │
│  └──────┬───────┘                                                   │
│         │ 위임                                                       │
│         ▼                                                            │
│  ┌──────────────────────┐                                            │
│  │ find_trending         │  Agent: finder + SerperDevTool            │
│  │ _companies            │  Output: TrendingCompanyList (Pydantic)   │
│  │                      │  → trending_companies.json                 │
│  └──────────┬───────────┘                                            │
│             │ context                                                 │
│             ▼                                                         │
│  ┌──────────────────────┐                                             │
│  │ research_trending     │  Agent: researcher + SerperDevTool         │
│  │ _companies            │  Output: TrendingCompanyResearchList       │
│  │                      │  → research_report.json                     │
│  └──────────┬───────────┘                                             │
│             │ context                                                  │
│             ▼                                                          │
│  ┌──────────────────────┐                                              │
│  │ pick_best_company     │  Agent: picker + PushNotificationTool       │
│  │                      │  1. 최적 기업 선정                           │
│  │                      │  2. 푸시 알림 전송                           │
│  │                      │  3. 상세 리포트 작성                         │
│  │                      │  → decision.md                               │
│  └──────────────────────┘                                              │
│                                                                        │
└────────────────────────────────────────────────────────────────────────┘
```

---

## 7. Stock Picker Crew 직접 실행

> **참고**: `SERPER_API_KEY`가 없으면 웹 검색 없이, `PUSHOVER_USER`/`PUSHOVER_TOKEN`이 없으면 알림 없이 실행됩니다.

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=SyntaxWarning, module="pysbd")

from dotenv import load_dotenv
import os

load_dotenv(override=True)

print(f"OPENAI_API_KEY:  {'found' if os.getenv('OPENAI_API_KEY') else 'NOT FOUND'}")
print(f"SERPER_API_KEY:  {'found' if os.getenv('SERPER_API_KEY') else 'NOT FOUND — 웹 검색 비활성화'}")
print(f"PUSHOVER_USER:   {'found' if os.getenv('PUSHOVER_USER') else 'NOT FOUND — 푸시 알림 비활성화'}")
print(f"PUSHOVER_TOKEN:  {'found' if os.getenv('PUSHOVER_TOKEN') else 'NOT FOUND — 푸시 알림 비활성화'}")

In [ ]:
from crewai import Agent, Task, Crew, Process
from pydantic import BaseModel, Field
from typing import List
from datetime import datetime

# ── Structured Output 모델 정의 ──

class TrendingCompany(BaseModel):
    name: str = Field(description="Company name")
    ticker: str = Field(description="Stock ticker symbol")
    reason: str = Field(description="Reason this company is trending in the news")

class TrendingCompanyList(BaseModel):
    companies: List[TrendingCompany] = Field(description="List of companies trending in the news")

class TrendingCompanyResearch(BaseModel):
    name: str = Field(description="Company name")
    market_position: str = Field(description="Current market position and competitive analysis")
    future_outlook: str = Field(description="Future outlook and growth prospects")
    investment_potential: str = Field(description="Investment potential and suitability for investment")

class TrendingCompanyResearchList(BaseModel):
    research_list: List[TrendingCompanyResearch] = Field(description="Comprehensive research on all trending companies")

print("Pydantic 모델 정의 완료")

In [ ]:
# ── 도구 준비 ──

finder_tools = []
researcher_tools = []
picker_tools = []

if os.getenv('SERPER_API_KEY'):
    from crewai_tools import SerperDevTool
    finder_tools = [SerperDevTool()]
    researcher_tools = [SerperDevTool()]
    print("SerperDevTool 활성화")
else:
    print("SerperDevTool 비활성화 — LLM 내부 지식만 사용")

# PushNotificationTool — 간단한 인라인 버전 (Pushover 없이도 동작)
from crewai.tools import BaseTool
from pydantic import BaseModel as PydanticBaseModel

class PushInput(PydanticBaseModel):
    message: str = Field(..., description="The message to be sent to the user.")

class PushNotificationTool(BaseTool):
    name: str = "Send a Push Notification"
    description: str = "This tool is used to send a push notification to the user."
    args_schema: type[PydanticBaseModel] = PushInput

    def _run(self, message: str) -> str:
        print(f"\n📱 Push Notification: {message}\n")
        # Pushover API 호출은 생략 (키가 있으면 실제 전송 가능)
        return '{"notification": "ok"}'

picker_tools = [PushNotificationTool()]
print("PushNotificationTool 준비 완료")

In [ ]:
# ── Agent 정의 ──

sector = "Technology"
current_date = str(datetime.now())

finder = Agent(
    role=f"Financial News Analyst that finds trending companies in {sector}",
    goal=f"You read the latest news, then find 2-3 companies that are trending in the news for further research. Always pick new companies.",
    backstory="You are a market expert with a knack for picking out the most interesting companies based on latest news.",
    verbose=True,
    llm="openai/gpt-4o-mini",
    tools=finder_tools,
    memory=True,
)

researcher = Agent(
    role="Senior Financial Researcher",
    goal="Given details of trending companies in the news, you provide comprehensive analysis of each in a report.",
    backstory="You are a financial expert with a proven track record of deeply analyzing hot companies and building comprehensive reports.",
    verbose=True,
    llm="openai/gpt-4o-mini",
    tools=researcher_tools,
)

picker = Agent(
    role="Stock Picker from Research",
    goal="Given a list of researched companies with investment potential, you select the best one for investment, notifying the user and then providing a detailed report.",
    backstory="You're a meticulous, skilled financial analyst with a proven track record of equity selection.",
    verbose=True,
    llm="openai/gpt-4o-mini",
    tools=picker_tools,
    memory=True,
)

manager = Agent(
    role="Manager",
    goal="You are a skilled project manager who can delegate tasks in order to achieve your goal, which is to pick the best company for investment.",
    backstory="You are an experienced and highly effective project manager who can delegate tasks to the right people.",
    llm="openai/gpt-4o-mini",  # 노트북에서는 비용 절약을 위해 mini 사용
    allow_delegation=True,
)

print(f"에이전트 4개 생성 완료: {finder.role[:30]}..., {researcher.role}, {picker.role}, {manager.role}")

In [ ]:
# ── Task 정의 (Structured Output 포함) ──

find_task = Task(
    description=f"Find the top trending companies in the news in {sector} by searching the latest news. Find new companies that you've not found before.",
    expected_output=f"A list of trending companies in {sector}",
    agent=finder,
    output_pydantic=TrendingCompanyList,       # ← Structured Output!
)

research_task = Task(
    description="Given a list of trending companies, provide detailed analysis of each company in a report by searching online",
    expected_output="A report containing detailed analysis of each company",
    agent=researcher,
    context=[find_task],                        # ← 명시적 context
    output_pydantic=TrendingCompanyResearchList, # ← Structured Output!
)

pick_task = Task(
    description=(
        "Analyze the research findings and pick the best company for investment. "
        "Send a push notification to the user with the decision and 1 sentence rationale. "
        "Then respond with a detailed report on why you chose this company."
    ),
    expected_output="The chosen company and why it was chosen; the companies that were not selected and why.",
    agent=picker,
    context=[research_task],                    # ← 명시적 context
)

print("태스크 3개 생성 완료 (find → research → pick)")
print(f"  find_task.output_pydantic = {find_task.output_pydantic.__name__}")
print(f"  research_task.output_pydantic = {research_task.output_pydantic.__name__}")
print(f"  pick_task: 자유 텍스트 출력")

In [ ]:
# ── Crew 구성 및 실행 (Hierarchical + Memory) ──

stock_crew = Crew(
    agents=[finder, researcher, picker],
    tasks=[find_task, research_task, pick_task],
    process=Process.hierarchical,
    manager_agent=manager,
    verbose=True,
    memory=True,
)

result = stock_crew.kickoff()

print("\n" + "="*60)
print("FINAL DECISION")
print("="*60)
print(result.raw)

In [ ]:
# ── 각 태스크 결과 확인 ──

from rich.console import Console
from rich.panel import Panel
from rich.markdown import Markdown
import json

console = Console()

titles = ["Trending Companies (Structured)", "Research Report (Structured)", "Investment Decision"]
colors = ["cyan", "yellow", "green"]

for i, task_output in enumerate(result.tasks_output):
    # 구조화된 출력이면 JSON으로 표시
    if task_output.pydantic:
        content = json.dumps(task_output.pydantic.model_dump(), indent=2, ensure_ascii=False)
    else:
        content = task_output.raw

    console.print(Panel(
        content,
        title=titles[i],
        border_style=colors[i],
    ))
    console.print()

---

## 정리

### Stock Picker 프로젝트에서 배운 것

```
┌─────────────────────────────────────────────────────────────────────┐
│              Stock Picker 핵심 요약                                 │
├──────────────────────────────┬──────────────────────────────────────┤
│     개념                     │     적용                             │
├──────────────────────────────┼──────────────────────────────────────┤
│  Structured Output           │  output_pydantic=TrendingCompanyList │
│  Custom Tool                 │  PushNotificationTool (외부 API)    │
│  Hierarchical Process        │  Manager(gpt-4o)가 동적 위임        │
│  Short-term Memory           │  RAG 기반, 현재 실행 중 컨텍스트    │
│  Long-term Memory            │  SQLite, 세션 간 영구 저장          │
│  Entity Memory               │  RAG 기반, 핵심 엔티티 추적         │
└──────────────────────────────┴──────────────────────────────────────┘
```

### Memory 타입 빠른 요약

| Memory | 한 줄 요약 | 비유 |
|--------|----------|------|
| **Short-term** | 현재 회의 중 공유하는 정보 | 화이트보드 |
| **Long-term** | 과거 경험에서 배운 것 | 업무 일지 |
| **Entity** | 주요 대상에 대한 핵심 정보 | 고객 DB |
| **Contextual** | 사용자의 취향/선호 학습 | 단골 고객 파악 |

### CLI로 실행하기

```bash
cd agent_engineering/03_crew/stock_picker
uv sync
crewai run
# 또는: uv run stock_picker
```